# Chapter 7: Dimensionality Reduction

Many machine learning problems involve training instances with thousands or even millions of features. This not only makes training extremely slow but also makes it harder to find a good solution.

## 1. The Curse of Dimensionality
In high-dimensional spaces, our intuition fails. 
*   **Sparsity:** In spaces with thousands of dimensions, data points are overwhelmingly likely to be located very far away from each other. 
*   **Overfitting:** Because training instances are far apart, predictions on new data are much less reliable. Models are more prone to overfitting because they try to extrapolate patterns across vast, empty spaces.

Dimensionality reduction helps cure this curse by compressing the data, speeding up training, and allowing us to visualize high-dimensional data in 2D or 3D graphs.

## 2. Main Approaches to Dimensionality Reduction

### A. Projection
In real-world problems, training instances are rarely spread out uniformly across all dimensions. They often lie close to a much lower-dimensional subspace. 
*   *Concept:* Imagine a 3D dataset where all points lie almost flat. We can simply drop the third dimension and "project" the points perpendicularly onto a 2D plane (like casting a shadow on a wall).

### B. Manifold Learning
Projection doesn't work for all datasets. The famous **Swiss Roll** dataset is a prime example. 
*   *Concept:* A Swiss Roll is a 2D plane that has been rolled up in 3D space. If we simply project it onto a plane, the different layers of the roll will squash together, destroying the pattern. 
*   Instead of squashing, Manifold Learning algorithms attempt to **unroll** the dataset to reveal its true, lower-dimensional shape.

## 3. The Manifold Assumption

Many dimensionality reduction algorithms work by modeling the **manifold** on which the training instances lie; this is called Manifold Learning.

*   **What is a Manifold?** Put simply, a 2D manifold is a 2D shape that has been bent and twisted in a higher-dimensional space (like our 3D Swiss Roll). Locally, it still behaves like a 2D plane.
*   **The Manifold Hypothesis:** This states that most real-world high-dimensional datasets lie close to a much lower-dimensional manifold. 
    * *Example:* If you randomly generate 784 pixels, you will get static noise. True MNIST handwritten digits only occupy a ridiculously tiny, specific fraction of that 784D space. They form a lower-dimensional manifold!
*   **The Goal:** We assume that our machine learning task (like classification or regression) will be much simpler if we first unroll this manifold and express the data in its true, lower-dimensional space.

## 4. The Manifold Assumption Pitfall

Does reducing the dimensionality of a dataset *always* make a machine learning task simpler? 
**No.** 

As shown in the bottom row of Figure 7-6, sometimes the decision boundary is very simple in the original high-dimensional space, but becomes highly complex and mangled when mapped to the lower-dimensional manifold space. 

### How to decide when to use Dimensionality Reduction?
There is no golden rule that tells you in advance whether dimensionality reduction will help or hurt your specific model. 
*   **The Solution:** Treat dimensionality reduction as an optional preprocessing step (a hyperparameter) in your Pipeline. You should always use **Cross-Validation** to compare your model's performance *with* and *without* dimensionality reduction. If the model performs better (or trains significantly faster with almost no loss in accuracy) with the reduced dataset, you keep it!

## Principal Component Analysis (PCA)

PCA is by far the most popular dimensionality reduction algorithm. It identifies the hyperplane that lies closest to the data, and then it projects the data onto it to reduce dimensions.

### 1. Principal Components
To project the data to a lower dimension, PCA first needs to find the right axes to project onto:
*   **First Principal Component (PC1):** The axis that accounts for the largest amount of variance in the training set. It preserves the maximum amount of information (think of it as casting the "widest shadow").
*   **Second Principal Component (PC2):** An axis orthogonal (at a 90-degree angle) to the first one, that accounts for the largest amount of the *remaining* variance.
*   If the dataset is 3D, PCA finds a third component (PC3) orthogonal to both PC1 and PC2, and so on for higher dimensions.

### 2. Preserving the Variance
When you are choosing the right hyperplane to project your data onto, you should choose the one that preserves the maximum variance. 
*   **Why?** Because it loses the least amount of information. 
*   In a 2D dataset, projecting the data onto the First Principal Component (PC1) will keep the points as spread out as possible. Projecting onto PC2 would squash the data points much closer together, destroying valuable patterns.

### 3. Dimensionality Reduction with PCA
Once PCA identifies all the Principal Components (equal to the number of dimensions in the dataset), we can reduce the dimensionality of our dataset down to $d$ dimensions. We do this by simply projecting the data onto the hyperplane defined by the first $d$ principal components (dropping the rest).

### 4. PCA in Practice: SVD and Scikit-Learn

Under the hood, PCA uses a standard matrix factorization technique called **Singular Value Decomposition (SVD)**. SVD decomposes the training set matrix $X$ into three matrices $U \cdot \Sigma \cdot V^T$. 
*   The matrix $V^T$ contains all the principal components (the axes we want to project onto).
*   **Important:** PCA assumes that the dataset is centered around the origin. If you implement PCA manually using SVD, you must explicitly center the data first (`X - X.mean(axis=0)`).

**The Scikit-Learn Way:**
Fortunately, Scikit-Learn’s `PCA` class takes care of everything for you, including the mean centering:

```python
from sklearn.decomposition import PCA

# Keep the top 2 principal components
pca = PCA(n_components=2)
X2D = pca.fit_transform(X) # Centers the data and projects it to 2D
```

### 5. Explained Variance Ratio
How do we know if reducing dimensions lost too much information? We look at the `explained_variance_ratio_` variable.

*   It tells you the proportion of the dataset's variance that lies along each principal component.
*   For example, if the output is `[0.82, 0.10]`, it means the first principal component (PC1) holds 82% of the original variance, and PC2 holds 10%. 
*   By projecting down to 2D, we preserved 92% of the variance and only lost 8% of the original information.

### 6. Choosing the Right Number of Dimensions

Instead of arbitrarily choosing the number of dimensions to reduce down to, it is generally preferable to choose the number of dimensions that add up to a sufficiently large portion of the variance (e.g., 95%).

**The Smart Way in Scikit-Learn:**
If you set `n_components` to be a float between 0.0 and 1.0, indicating the ratio of variance you wish to preserve, PCA will automatically calculate how many dimensions are needed to reach that target!

```python
# Keep as many dimensions as needed to preserve 95% of the variance
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X)
```
*Note: Unless you are reducing dimensionality for data visualization (where you are strictly forced to reduce down to 2 or 3 dimensions), this 95% rule is the standard approach.*

### 7. Real-World Example: PCA on MNIST

To see PCA in action on a larger dataset, let's look at the MNIST dataset (images of handwritten digits). Each image is 28x28 pixels, which means the dataset has **784 dimensions** (features).

First, let's load the data:
```python
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=False)
X_train, y_train = mnist.data[:60_000], mnist.target[:60_000]
X_test, y_test = mnist.data[60_000:], mnist.target[60_000:]
```

#### Method 1: Calculating Dimensions Manually
We can fit PCA on the training set, compute the cumulative sum of the explained variance ratio, and find the exact number of dimensions required to preserve 95% of the variance:

```python
import numpy as np
from sklearn.decomposition import PCA

pca = PCA()
pca.fit(X_train)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1  
# Output of d is 154
```
This tells us that out of 784 original pixels, only **154** are needed to keep 95% of the information!

#### Method 2: The Smart Way (Scikit-Learn feature)
Instead of doing the math manually, we can just tell Scikit-Learn to do it for us:
```python
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)

# Checking how many components it kept
pca.n_components_
# Output: 154
```

### 8. Visualizing Explained Variance (The Elbow Plot)

A great way to decide the number of dimensions is to plot the cumulative explained variance as a function of the number of dimensions. 
*   There will usually be an "elbow" in the curve, where the explained variance stops growing significantly. 
*   For the MNIST dataset (which originally has 784 dimensions), the curve flattens out around 150 dimensions. This tells us that the remaining ~600 dimensions add almost zero valuable information (they are likely just background pixels).


### 9. Tuning PCA as a Hyperparameter in a Pipeline

Dimensionality reduction is usually a preprocessing step. Instead of blindly choosing 95% variance, we can treat the number of components (`n_components`) as a hyperparameter and let cross-validation find the absolute best value for our specific machine learning model.

#### Example A: Tuning with Random Forest
We can create a `Pipeline` containing PCA and a `RandomForestClassifier`, and use `RandomizedSearchCV` to find the best combination:

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

clf = make_pipeline(PCA(random_state=42),
                    RandomForestClassifier(random_state=42))

param_distrib = {
    "pca__n_components": np.arange(10, 80),
    "randomforestclassifier__n_estimators": np.arange(50, 500)
}

rnd_search = RandomizedSearchCV(clf, param_distrib, n_iter=10, cv=3, random_state=42)
rnd_search.fit(X_train[:1000], y_train[:1000])

print(rnd_search.best_params_)
# Output: {'randomforestclassifier__n_estimators': 475, 'pca__n_components': 57}
```
*Result:* For the Random Forest model, keeping only **57 dimensions** yielded the highest accuracy!

#### Example B: Tuning with SGD Classifier
Different models require different amounts of information. Let's see what happens if we use an `SGDClassifier` and `GridSearchCV`:

```python
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV

clf = make_pipeline(PCA(random_state=42), SGDClassifier())

param_grid = {"pca__n_components": np.arange(10, 80)}
grid_search = GridSearchCV(clf, param_grid, cv=3)
grid_search.fit(X_train[:1000], y_train[:1000])

print(grid_search.best_params_)
# Output: {'pca__n_components': 68}
```
*Result:* For the SGD Classifier, the optimal number of dimensions was **68**. This proves that `n_components` should be tuned based on the specific algorithm you are using.

### 10. PCA for Compression and Decompression

One of the most powerful applications of PCA is data compression. As we saw with the MNIST dataset, we can reduce the features from 784 down to 154 while preserving 95% of the variance. This means the compressed dataset takes up less than 20% of its original size! This is incredibly useful for speeding up algorithms and saving storage space.

#### The Magic of `inverse_transform`
But what if you want to look at the compressed images again? You can project the reduced dataset (154 dimensions) back into the original high-dimensional space (784 dimensions). 

Scikit-Learn makes this decompression process easy with the `inverse_transform()` method:

```python
# 1. Compress the data (keep 95% variance)
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train) # Goes from 784 to 154 dimensions

# 2. Decompress the data back to the original space
X_recovered = pca.inverse_transform(X_reduced) # Goes from 154 back to 784 dimensions
```

#### Reconstruction Error
If you plot the original images next to the `X_recovered` images, you will notice a difference. The recovered images look a bit blurrier and have lost some of their crispness. 

*   **Why does this happen?** Because we intentionally dropped 5% of the variance during the compression phase. That 5% contained the fine details, sharp edges, and noise of the images. 
*   Once that information is dropped, it is permanently lost; `inverse_transform` cannot magically bring it back. 
*   The mean squared distance between the original data ($X$) and the reconstructed data (`X_recovered`) is called the **Reconstruction Error**.

### 11. Randomized PCA

Standard PCA (using the full SVD approach) can be extremely slow when applying it to high-dimensional datasets. This is because it calculates *all* principal components, even if we only plan to keep a small fraction of them.

To fix this, Scikit-Learn offers a stochastic algorithm called **Randomized PCA** that quickly finds an approximation of the first $d$ principal components. 

*   **The Benefit:** Its computational complexity is significantly lower than standard PCA. It is dramatically faster when the number of dimensions you want to keep ($d$) is much smaller than the original number of dimensions ($n$).
*   **How to use it:** Simply set the `svd_solver` hyperparameter to `"randomized"`.

```python
from sklearn.decomposition import PCA
import time

# Start the timer to see how fast it is
t0 = time.time()

# Use Randomized PCA
rnd_pca = PCA(n_components=154, svd_solver="randomized", random_state=42)
X_reduced_rnd = rnd_pca.fit_transform(X_train)

t1 = time.time()
print(f"Training took {t1 - t0:.2f} seconds")
```

*(Note: By default, `svd_solver` is actually set to `"auto"`. Scikit-Learn is smart enough to automatically switch to the randomized algorithm if the dataset is large enough and the target components are small enough.)*

### 12. Incremental PCA (IPCA)

Standard PCA requires the entire training set to fit in memory (RAM) in order for the algorithm to run. Fortunately, **Incremental PCA (IPCA)** algorithms have been developed to solve this out-of-memory (OOM) problem. 

IPCA allows you to split the training set into mini-batches and feed an IPCA algorithm one mini-batch at a time. This is useful for large training sets and for applying PCA online (i.e., on the fly, as new data arrives).

#### Method 1: Using `partial_fit`
Instead of using the standard `.fit()` method, we can loop over our mini-batches and use the `.partial_fit()` method to update the PCA incrementally:

```python
from sklearn.decomposition import IncrementalPCA
import numpy as np

n_batches = 100
inc_pca = IncrementalPCA(n_components=154)

# Splitting the data into 100 mini-batches
for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch) # Updates the model incrementally

X_reduced = inc_pca.transform(X_train)
```

#### Method 2: Using NumPy's `memmap` (Out-of-Core Learning)
If the dataset is too large to even load into memory to split it (e.g., a 100GB file), you can use NumPy's `memmap` class. It allows you to manipulate a large array stored in a binary file on disk as if it were entirely in memory; it loads only the data it needs right now.

```python
# 1. Load the data using memmap (reads directly from disk, not RAM)
filename = "my_mnist.mmap"
X_mmap = np.memmap(filename, dtype='float32', mode='readonly', shape=(-1, 784))

# 2. Use IncrementalPCA with a specific batch_size
batch_size = X_mmap.shape[0] // n_batches
inc_pca = IncrementalPCA(n_components=154, batch_size=batch_size)

# 3. Call fit() directly on the memmap array
inc_pca.fit(X_mmap)
```
*Note: Because `IncrementalPCA` supports the `batch_size` parameter, you can just call `.fit()` directly on the `memmap` array, and Scikit-Learn will handle the mini-batching under the hood!*

### 13. Random Projection

**Random Projection** is fundamentally different from PCA. While PCA searches for the optimal axes that preserve the most variance, Random Projection does not look at the data's variance at all. It simply projects the data onto completely random axes!

#### The Johnson-Lindenstrauss Lemma
How can projecting data onto random axes possibly work? The mathematical foundation is the **Johnson-Lindenstrauss lemma**. It states that if you project points from a very high-dimensional space into a suitably chosen lower-dimensional space using random axes, the distances between the points are approximately preserved.

If distances are preserved, machine learning models (like SVMs or Clustering algorithms) can still easily recognize patterns and separate the data!

#### Determining the Target Dimensions ($d$)
We don't need to guess how many dimensions we need. We can calculate the minimum number of dimensions $d$ required to guarantee that distances change by no more than a tolerance $\varepsilon$ (epsilon):

```python
from sklearn.random_projection import johnson_lindenstrauss_min_dim

m, epsilon = 5_000, 0.1 # 5,000 instances, max 10% distortion in distances
d = johnson_lindenstrauss_min_dim(m, eps=epsilon)
# Output: 7300
```

#### Gaussian vs. Sparse Random Projection
Scikit-Learn provides two main classes for this:
1.  **`GaussianRandomProjection`**: Creates a random projection matrix where values are drawn from a Gaussian distribution.
2.  **`SparseRandomProjection`**: A much more efficient alternative. The random matrix it generates is sparse (mostly filled with zeros). This requires drastically less memory and makes the matrix multiplication much faster, while providing almost the exact same quality of dimensionality reduction.

```python
from sklearn.random_projection import SparseRandomProjection

# It is generally recommended to use SparseRandomProjection for massive datasets
sparse_rnd_proj = SparseRandomProjection(random_state=42)
X_reduced = sparse_rnd_proj.fit_transform(X)
```

### 14. Manifold Learning: LLE

When Projection algorithms (like PCA) fail on complex, twisted datasets (like the Swiss Roll), we turn to **Manifold Learning**. 

**Locally Linear Embedding (LLE)** is a powerful nonlinear dimensionality reduction (NLDR) technique. Unlike PCA, which tries to preserve global variance, LLE focuses entirely on preserving **local relationships**.

#### How LLE Works (in 2 steps):
1.  **Measure locally:** For each training instance, it identifies its $k$ nearest neighbors (e.g., $k=10$) and evaluates how this instance is linearly related to them.
2.  **Unroll:** It then looks for a lower-dimensional space where these local linear relationships are best preserved. 

This local focus makes LLE exceptionally good at unrolling twisted manifolds without squashing the layers together.

```python
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import LocallyLinearEmbedding
import matplotlib.pyplot as plt

# Generate a 3D Swiss Roll dataset
X_swiss, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)

# Apply LLE to unroll it to 2D
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, random_state=42)
X_unrolled = lle.fit_transform(X_swiss)

# Plotting the result
plt.scatter(X_unrolled[:, 0], X_unrolled[:, 1], c=t, cmap=plt.cm.hot)
plt.title("Unrolled swiss roll using LLE")
plt.show()
```
*Result:* As seen in the plot, the Swiss roll is perfectly unrolled, and the colors (representing the original layout) transition smoothly without overlapping.


### 15. Other Dimensionality Reduction Techniques

Scikit-Learn offers several other Manifold Learning algorithms for specific use cases:

*   **MDS (Multidimensional Scaling):** Reduces dimensionality while trying to preserve the distances between *all* instances (not just local neighbors).
*   **Isomap:** Creates a graph by connecting each instance to its nearest neighbors, then reduces dimensionality while trying to preserve the *geodesic distances* (the number of nodes on the shortest path) between instances on the graph.
*   **t-SNE (t-Distributed Stochastic Neighbor Embedding):** Reduces dimensionality while trying to keep similar instances close and dissimilar instances apart. It is mostly used for data visualization, especially to visualize clusters of instances in high-dimensional space.

## Extra Material: Kernel PCA

Standard PCA is a linear algorithm. However, by using the **kernel trick** (a mathematical technique that implicitly maps instances into a very high-dimensional space called the feature space), we can perform complex nonlinear projections for dimensionality reduction. This is called **Kernel PCA (kPCA)**.

It is often good at preserving clusters of instances after projection, or sometimes even unrolling datasets that lie close to a twisted manifold.

### Visualizing Different Kernels
Scikit-Learn makes it very simple to apply kPCA using the `KernelPCA` class. You can experiment with different kernels (like Linear, RBF, and Sigmoid) to see how they unravel complex data like the Swiss roll.

```python
from sklearn.decomposition import KernelPCA
import matplotlib.pyplot as plt

# 1. Linear Kernel (Equivalent to standard PCA)
lin_pca = KernelPCA(n_components=2, kernel="linear")

# 2. RBF Kernel (Non-linear)
rbf_pca = KernelPCA(n_components=2, kernel="rbf", gamma=0.002, random_state=42)

# 3. Sigmoid Kernel (Non-linear)
sig_pca = KernelPCA(n_components=2, kernel="sigmoid", gamma=0.002, coef0=1, random_state=42)

# Fit and transform the data (example with RBF)
X_reduced_rbf = rbf_pca.fit_transform(X_swiss)
```
*   **Linear Kernel:** Will simply squash the manifold, mixing the layers.
*   **Non-linear Kernels (RBF/Sigmoid):** Can successfully separate the layers of the twisted manifold, making downstream classification or clustering much easier.

### 16. Tuning Kernel PCA Hyperparameters

Because `KernelPCA` is an unsupervised learning algorithm, there is no obvious objective performance measure to help you select the best kernel and hyperparameter values (like $\gamma$). 

However, dimensionality reduction is often a preparation step for a supervised learning task (e.g., classification). Therefore, you can simply use grid search to select the kernel and hyperparameters that lead to the best performance on that final supervised task!

```python
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import numpy as np

# 1. Create a pipeline with kPCA and a Classifier
clf = Pipeline([
    ("kpca", KernelPCA(n_components=2)),
    ("log_reg", LogisticRegression())
])

# 2. Define the hyperparameter grid to search
param_grid = [{
    "kpca__gamma": np.linspace(0.03, 0.05, 10),
    "kpca__kernel": ["rbf", "sigmoid"]
}]

# 3. Run Grid Search
grid_search = GridSearchCV(clf, param_grid, cv=3)
grid_search.fit(X, y) # X and y are your training data and labels

# 4. See the winning combination!
print(grid_search.best_params_)
```

## Chapter 7 Summary: Key Takeaways & Best Practices

Based on the end-of-chapter exercises, here is a consolidated summary of the most important rules for Dimensionality Reduction.

### 1. Motivations and Drawbacks (Q1, Q3, Q7)
*   **Motivations:** The main reasons to reduce dimensions are to speed up training, visualize complex data, and save storage space (compression).
*   **Drawbacks:** It always involves some information loss, which might degrade the performance of downstream models. It also makes pipelines more complex and features harder to interpret.
*   **Reversibility:** You can never perfectly reverse the operation because of the lost information (Reconstruction Error). Some algorithms (like PCA) have an `inverse_transform` method, while others (like t-SNE) do not.
*   **Evaluation:** To evaluate a reduction algorithm, you can either measure its reconstruction error or measure the accuracy of the final supervised model (e.g., Random Forest) with and without the reduction step.

### 2. The Curse and Non-linear Data (Q2, Q4, Q5)
*   **Curse of Dimensionality:** In high-dimensional spaces, data points are extremely sparse and far apart. This massively increases the risk of overfitting unless you have an astronomically large dataset.
*   **PCA on Non-linear Data:** PCA can still be useful on highly non-linear datasets if it helps get rid of completely useless dimensions. However, it will fail (squash the data) if the non-linearity is a complex manifold like a Swiss roll.
*   **The 95% Variance Trick:** If you apply `PCA(n_components=0.95)` to a 1000D dataset, the resulting number of dimensions depends entirely on the dataset. It could be 1 dimension (if data is perfectly aligned) or 950 dimensions (if the data is completely random and scattered).

### 3. Choosing the Right Algorithm (Q6, Q8)
*   **Regular PCA:** The default choice if the dataset fits in your computer's RAM.
*   **Incremental PCA:** The go-to solution for massive datasets that do not fit in memory, or for online learning tasks (streaming data).
*   **Randomized PCA:** Use this when the dataset fits in memory but you want to significantly speed up the calculation to find the top principal components.
*   **Random Projection:** Best suited for extremely high-dimensional datasets.
*   **Chaining Algorithms:** It is a highly effective strategy to chain algorithms! For example, you can use PCA or Random Projection first to quickly discard useless dimensions, and then apply a slower, more complex algorithm like LLE to unroll the manifold.

# Chapter 7 Summary: Dimensionality Reduction (Cheat Sheet)

## 1. Core Concepts & The Curse of Dimensionality
*   **The Curse:** In high-dimensional spaces, data points are extremely sparse. This increases the risk of overfitting and makes training computationally expensive.
*   **Main Motivations:** Speed up training, compress data (save space), and visualize complex datasets in 2D or 3D.
*   **The Trade-off:** Dimensionality reduction *always* causes some information loss. It can degrade the performance of downstream models and make pipelines more complex.
*   **Two Main Approaches:** 
    1.  **Projection:** Projects data onto a lower-dimensional hyperplane (e.g., PCA, Random Projection). Fails on twisted datasets.
    2.  **Manifold Learning:** Unrolls twisted, complex datasets by measuring how each training instance relates to its closest neighbors (e.g., LLE, t-SNE).

## 2. Principal Component Analysis (PCA)
*   **How it works:** PCA identifies the hyperplane that preserves the maximum amount of **variance** (minimizes information loss/reconstruction error).
*   **Under the hood:** Uses Singular Value Decomposition (SVD). The data *must* be centered around the origin first (Scikit-Learn's `PCA` does this automatically).
*   **Choosing Dimensions ($d$):** Instead of guessing $d$, set `n_components` to a float (e.g., `0.95`) to preserve 95% of the variance. Alternatively, plot the explained variance to find the "elbow".
*   **Compression & Decompression:** You can compress data using `fit_transform()` and decompress it using `inverse_transform()`. The difference between the original and decompressed data is the **Reconstruction Error**.

## 3. Scaling PCA for Large Datasets
*   **Randomized PCA:** Uses stochastic algorithms to quickly approximate the first $d$ principal components. Drastically faster than standard PCA when $d$ is much smaller than the original features (`svd_solver="randomized"`).
*   **Incremental PCA (IPCA):** Solves the Out-Of-Memory (OOM) problem. It splits massive datasets into mini-batches and updates the model iteratively using `partial_fit()`, or by reading directly from the disk using `np.memmap`.

## 4. Random Projection
*   **The Concept:** Does not look at the data's variance. Instead, it projects data onto completely *random axes*. 
*   **Johnson-Lindenstrauss Lemma:** Guarantees that if the target dimensionality is large enough, the distances between data points are approximately preserved.
*   **Performance:** Incredibly fast and memory-efficient, making it ideal for datasets with massive dimensionality. Use `SparseRandomProjection` for optimal performance.

## 5. Nonlinear Dimensionality Reduction
*   **LLE (Locally Linear Embedding):** A Manifold Learning technique. It preserves *local* distances (how a point relates to its $k$ nearest neighbors) rather than global distances. Perfect for unrolling a Swiss Roll.
*   **Kernel PCA (kPCA):** Uses the "kernel trick" to implicitly map data into a high-dimensional space to perform complex, non-linear projections (using kernels like RBF or Sigmoid).

## 6. Best Practices in the Real World
1.  **Pipeline Tuning:** Treat the number of dimensions (`n_components`) and hyperparameters (like kPCA's `gamma` or `kernel`) as standard hyperparameters. Put the reduction algorithm in a `Pipeline` and use `GridSearchCV` to find the combination that maximizes the accuracy of your final classification/regression model.
2.  **Chaining Algorithms:** You can chain multiple algorithms for efficiency! For example, use PCA or Random Projection to quickly discard useless dimensions, and then apply a slower, more accurate algorithm like LLE on the reduced dataset.